In [ ]:
# Cell 1 - Imports
import os
from dotenv import load_dotenv
from llama_index.core import StorageContext, load_index_from_storage
from llama_index.core import Settings

import sys
# setting path
# Aggiungi la cartella PARENT della cartella Parla_con_PG_TM
project_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# import from parent directory llm.py:
from Parla_con_PG_TM.llm import init_local_embed_model


load_dotenv()

True

In [ ]:
# Cell 3 - Load existing vector store
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings, StorageContext, load_index_from_storage
PERSIST_DIR = project_root + "\Parla_con_PG_TM\chroma_db"

def load_vector_store():
    """Load the existing vector store from disk"""
    if not os.path.exists(PERSIST_DIR):
        raise ValueError(f"Storage directory '{PERSIST_DIR}' not found")
    Settings.embed_model = init_local_embed_model()
    print("Loading vector store...")
    chroma_client = chromadb.PersistentClient(path=PERSIST_DIR)
    collection = chroma_client.get_collection(name="ardania_lore")
    vector_store = ChromaVectorStore(chroma_collection=collection)
    index = VectorStoreIndex.from_vector_store(vector_store=vector_store)
    print("Vector store loaded successfully")
    return index

In [3]:

# Cell 4 - Query function
def query_similar_docs(index, query_text, top_k=3):
    """
    Retrieve top k most similar documents to the query
    """
    # Create query embedding and retrieve similar docs
    retriever = index.as_retriever(similarity_top_k=top_k)
    nodes = retriever.retrieve(query_text)
    
    print(f"\nTop {top_k} similar documents to '{query_text}':\n")
    for i, node in enumerate(nodes, 1):
        print(f"Document {i}:")
        print(f"Score: {node.score:.4f}")
        print(f"Content: {node.text}\n")
    
    return nodes

In [7]:
# Cell 5 - Execute query
if __name__ == "__main__":
    # Load index
    index = load_vector_store()
    
    # Query for similar documents
    query_text = "il chierico"
    similar_docs = query_similar_docs(index, query_text)

Loading vector store...
Vector store loaded successfully

Top 3 similar documents to 'il chierico':

Document 1:
Score: 0.0000
Content: ne e lecucciolate. Molto ben nascoste. Lo fanno per evitare guai. E pensa che c’è qualche esploratore che… beh,è un po’ strano a dirsi, ma ho sentito di un pazzo che va in giro a dire che, in realtà, ci sia una sola femmina.Una! Hai capito sì?Comunque, al netto di ogni baggianata possa uscire dalla bocca di qualche topo di biblioteca… in realtà,sono abbastanza utili. E non parlo solo di pellicce o pelli: quelle bestie fanno il lavoro più ingrato e infamedella natura. In poche parole, tengono pul

Document 2:
Score: 0.0000
Content: tanze che uscivano dalla cucina che dalle noteemesse dal suo strumento. Tutto intorno un vociare costante ma non invadente coloriva l’atmosfera di un’anonima locandadella costa. Al passaggio dell’ennesimo sformato, caldo e profumato, il brontolio di stomaco che ne seguì fu troppo forteper poter essere ignorato e il ragazzo de

In [6]:
# print all the scores of similar documents
print("\nScores of similar documents:")
for i, doc in enumerate(similar_docs, 1):
    print(f"Document {i} Score: {doc.score:.4f}")
# End of script


Scores of similar documents:
Document 1 Score: 0.5162
Document 2 Score: 0.5155
Document 3 Score: 0.5055
